# MNIST CNN Inference on PYNQ-Z2
Run this notebook **on the PYNQ-Z2 board** (Jupyter at 192.168.2.99)

Requirements:
- `cnn.bit` + `cnn.hwh` in the same directory as this notebook
- `fpga_weights/` folder (w_conv1.npy, b_conv1.npy, etc.) in the same directory
- MNIST test set available (downloads automatically)


In [ ]:
import numpy as np
import time
from pynq import Overlay, allocate
import pynq.lib.dma
import urllib.request, gzip, os

# Download MNIST test set for evaluation
def download_mnist():
    base = 'http://yann.lecun.com/exdb/mnist/'
    files = ['t10k-images-idx3-ubyte.gz', 't10k-labels-idx1-ubyte.gz']
    for f in files:
        if not os.path.exists(f[:-3]):
            print(f'Downloading {f}...')
            urllib.request.urlretrieve(base + f, f)
            with gzip.open(f) as gz, open(f[:-3],'wb') as out:
                out.write(gz.read())
    imgs  = np.frombuffer(open('t10k-images-idx3-ubyte','rb').read(), np.uint8, offset=16).reshape(-1,784)
    labels= np.frombuffer(open('t10k-labels-idx1-ubyte','rb').read(), np.uint8, offset=8)
    return imgs, labels

imgs, labels = download_mnist()
print(f'Loaded {len(imgs)} test images')


In [ ]:
# Load the FPGA overlay (bitstream + hardware handoff)
print('Loading overlay...')
ol = Overlay('./cnn.bit')
print('Overlay loaded.')

dma = ol.axi_dma_0
cnn = ol.cnn_top_0


In [ ]:
# Load quantised weights (int16) and allocate contiguous DDR buffers
SCALE = 1024   # matches quantisation scale in export script

weight_names = ['w_conv1', 'b_conv1', 'w_conv2', 'b_conv2', 'w_fc', 'b_fc']
bufs = {}

for name in weight_names:
    arr = np.load(f'fpga_weights/{name}.npy')
    buf = allocate(shape=arr.shape, dtype=np.int16)
    buf[:] = arr
    bufs[name] = buf
    print(f'Loaded {name}: shape {arr.shape}, physical_address 0x{buf.physical_address:08X}')

# AXI-Lite register map for weight pointers (from xcnn_top_hw.h)
REG = {
    'w_conv1': 0x10, 'b_conv1': 0x1c,
    'w_conv2': 0x28, 'b_conv2': 0x34,
    'w_fc': 0x40,    'b_fc': 0x4c,
}

# Write physical addresses to the CNN accelerator's control registers
for name in weight_names:
    cnn.write(REG[name], bufs[name].physical_address & 0xFFFFFFFF)

print('Weight addresses configured in hardware.')


In [ ]:
def preprocess(img_uint8):
    x = img_uint8.astype(np.float32) / 255.0
    x = (x - 0.1307) / 0.3081
    return np.clip(np.round(x * SCALE), -32768, 32767).astype(np.int16)

in_buf  = allocate(shape=(784,), dtype=np.int16)
out_buf = allocate(shape=(10,),  dtype=np.int16)

def fpga_inference(img_uint8):
    in_buf[:] = preprocess(img_uint8)
    
    # AP_START (0x00) starts the accelerator
    cnn.write(0x00, 1)
    
    # Stream inputs and outputs
    dma.sendchannel.transfer(in_buf)
    dma.recvchannel.transfer(out_buf)
    dma.sendchannel.wait()
    dma.recvchannel.wait()
    
    return out_buf[:].astype(np.float32) / SCALE


In [ ]:
idx = 0
logits = fpga_inference(imgs[idx])
pred = np.argmax(logits)
print(f'Prediction: {pred} (True label: {labels[idx]})')
print(f'Logits: {np.round(logits, 2)}')


In [ ]:
N = 1000
correct = 0
t0 = time.time()

for i in range(N):
    logits = fpga_inference(imgs[i])
    if np.argmax(logits) == labels[i]:
        correct += 1
        
elapsed = time.time() - t0
print(f'=== CNN FPGA Benchmark ({N} images) ===')
print(f'Accuracy    : {correct/N*100:.2f}%')
print(f'Throughput  : {N/elapsed:.1f} images/sec')
